# Experiment 3 — Data Cleaning, Preprocessing, Filtering & Storage (SQLite + MongoDB)
### Social Media Analytics Lab

This notebook performs data cleaning, preprocessing, filtering, and database storage on three scraped datasets (all in **CSV** format), one after another:

1. **Instagram data** (`instagram_dataset.csv`)
2. **YouTube data** (`youtube_data.csv`)
3. **Amazon data** (`amazon_dataset.csv`)

For each dataset we will:
- Load the CSV and inspect its shape
- Clean the text fields (remove whitespace, punctuation, HTML tags, URLs, repeated letters, stopwords, and convert to lowercase)
- Remove duplicate records
- Apply a filter to work with a meaningful subset of the data
- Store the cleaned data into **both** a **SQLite** (relational) database file **and** a **MongoDB** (non-relational) collection running on your PC

---
### About SQLite vs. MongoDB in this notebook

- **SQLite** needs no server or installation at all — it's just a `.db` file created directly by Python, so it works fine in Google Colab as well as locally.
- **MongoDB**, however, is a server running **on your own PC**. Google Colab runs on Google's remote servers, so it **cannot** reach `localhost:27017` on your machine directly.

So: if you're running this in **Google Colab**, the SQLite parts will work as-is, but the MongoDB parts will fail to connect unless you expose your local MongoDB to the internet (e.g. via `ngrok`), which is extra setup you probably don't need for a lab experiment.

**Simplest path:** download this `.ipynb` and run it locally in Jupyter Notebook / JupyterLab / Anaconda on your own PC, where both SQLite and MongoDB (`localhost:27017`) will work correctly without any extra setup.


## 0. Setup — install and import libraries

If running locally and these are already installed, you can skip/re-run this cell safely.

In [33]:
# Install required libraries
!pip install nltk pandas pymongo -q


In [34]:
import re
import itertools
import pandas as pd

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

import sqlite3
from pymongo import MongoClient

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print("Libraries loaded successfully.")


Libraries loaded successfully.


## 0.1 Place your CSV files

Make sure the following three files are in the same folder as this notebook (or update the paths below):
- `instagram_dataset.csv`
- `youtube_data.csv`
- `amazon_dataset.csv`


In [35]:
INSTAGRAM_CSV = 'instagram_dataset.csv'
YOUTUBE_CSV   = 'youtube_data.csv'
AMAZON_CSV    = 'amazon_dataset_full.csv'   # full raw export, all original columns


## 0.2 Connect to a SQLite database

SQLite needs no setup at all — running the cell below simply creates a database file called `social_media.db` in the same folder as this notebook (or opens it if it already exists).

In [36]:
SQLITE_DB_PATH = 'social_media.db'

sqlite_conn = sqlite3.connect(SQLITE_DB_PATH)
print(f"Connected to SQLite database '{SQLITE_DB_PATH}'")


Connected to SQLite database 'social_media.db'


## 0.3 Connect to your local MongoDB database

Make sure your MongoDB server (`mongod`) is running on your PC before executing this cell — e.g. as a Windows/Mac service, or by running `mongod` in a terminal. By default it runs on port `27017` with no authentication required for local connections.

In [37]:
MONGO_HOST = 'localhost'
MONGO_PORT = 27017
MONGO_DB = 'social_media_analytics'

mongo_client = MongoClient(MONGO_HOST, MONGO_PORT, serverSelectionTimeoutMS=5000)

# Quick check that the connection actually works
try:
    mongo_client.admin.command('ping')
    print(f"Connected to MongoDB at {MONGO_HOST}:{MONGO_PORT}")
except Exception as e:
    print("Could not connect to MongoDB. Make sure 'mongod' is running on your PC.")
    raise e

mongo_db = mongo_client[MONGO_DB]


Connected to MongoDB at localhost:27017


## 1. Text cleaning function

This function follows the preprocessing pipeline discussed in the theory section of this experiment:
1. Strip leading/trailing whitespace
2. Remove HTML tags
3. Remove URLs
4. Remove punctuation
5. Standardize repeated letters (e.g. "happpppy" → "happy")
6. Convert to lowercase
7. Remove stopwords

We also define a separate function for stemming/lemmatization, which can optionally be applied on top of this.

In [38]:
def clean_text(text):
    #"""Cleans a single piece of text using the standard preprocessing pipeline."""
    if not isinstance(text, str) or text.strip() == '':
        return ''

    # 1. Remove whitespace
    text = text.strip()

    # 2. Remove HTML tags
    text = re.sub(r'<[^<]+?>', '', text)

    # 3. Remove URLs
    text = re.sub(r'https?://\S+', '', text)

    # 3.5 Remove emojis
    emoji_pattern = re.compile(
        '['
        u'\U0001F300-\U0001FAFF'   # symbols, pictographs, emoticons
        u'\U00002700-\U000027BF'   # dingbats
        u'\U0001F1E6-\U0001F1FF'   # flags
        u'\U00002600-\U000026FF'   # misc symbols
        u'\U0001F900-\U0001F9FF'   # supplemental symbols
        u'\U00002300-\U000023FF'   # technical symbols
        ']+', flags=re.UNICODE
    )
    text = emoji_pattern.sub('', text)

    # 4. Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # 5. Standardize words (remove multiple repeated letters, e.g. "happpppy" -> "happy")
    text = ''.join(''.join(s)[:2] for _, s in itertools.groupby(text))

    # 6. Convert to lowercase
    text = text.lower()

    # 7. Stop word removal
    text = ' '.join([word for word in text.split() if word not in stop_words])

    return text


def stem_and_lemmatize(text):
    #"""Applies stemming and lemmatization to already-cleaned text."""
    tokens = text.split()
    stemmed = [stemmer.stem(t) for t in tokens]
    lemmatized = [lemmatizer.lemmatize(t) for t in stemmed]
    return ' '.join(lemmatized)


# quick test
sample = "This productttt is Amazingg!! Check it out: https://example.com <b>Best ever</b>"
print("Original :", sample)
print("Cleaned  :", clean_text(sample))


Original : This productttt is Amazingg!! Check it out: https://example.com <b>Best ever</b>
Cleaned  : productt amazingg check best ever


---
## 2. Instagram Data — Cleaning, Filtering & Storage

### 2.1 Load and inspect the raw data

In [39]:
insta_df = pd.read_csv(INSTAGRAM_CSV)
print("Shape:", insta_df.shape)
insta_df.head(3)


Shape: (80, 42)


,id,type,shortCode,caption,hashtags,mentions,url,commentsCount,firstComment,latestComments,...,musicInfo.artist_name,musicInfo.song_name,musicInfo.uses_original_audio,musicInfo.should_mute_audio,musicInfo.should_mute_audio_reason,musicInfo.audio_id,videoPlayCount,isPinned,paidPartnership,sponsors
0,3934596544011844684,Video,DaafrXcTjxM,Every win has a story. Every story begins long...,NaN,NaN,https://www.instagram.com/p/DaafrXcTjxM/,188,Good promotion 😂,"[{""id"": ""17878068531506204"", ""text"": ""Good pro...",...,muscleblaze,Original audio,True,False,NaN,2.731969e+16,181012.0,NaN,NaN,NaN
1,3930923218166072046,Sidecar,DaNcdcKH8Lu,"Before he carried the hopes of a nation, \nhe ...",NaN,parthaesthetix,https://www.instagram.com/p/DaNcdcKH8Lu/,43,❤️❤️,"[{""id"": ""18095779619196592"", ""text"": ""❤️❤️"", ""...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3929463792106173155,Video,DaIQn_uP3Lj,The records made him known.\n\nThe pursuit mad...,NaN,NaN,https://www.instagram.com/p/DaIQn_uP3Lj/,60,@rizwan.pathan_oo7,"[{""id"": ""18109613371790047"", ""text"": ""@rizwan....",...,muscleblaze,Original audio,True,False,NaN,2.829017e+16,184946.0,NaN,NaN,NaN


### 2.2 Column filtering
Filtering doesn't only mean keeping a subset of *rows* — it can also mean keeping only the *columns* that are actually useful for analysis. Based on inspecting the data (checking for empty, constant, or purely technical fields), we drop:
- `inputUrl` - the URL originally given to the scraper, not a property of the post itself
- `dimensionsHeight`, `dimensionsWidth` - raw image pixel dimensions, not relevant to engagement analysis
- `musicInfo.should_mute_audio_reason` - 100% empty across all posts
- `isCommentsDisabled` - constant (always `False`), so it carries no information
- `displayUrl` - a raw CDN image link, not analytically useful
- `originalHeight`, `originalWidth` - image resolution, not engagement-related
- `alt` - Instagram's auto-generated accessibility text, redundant with the caption
- `ownerId` - redundant, since `ownerUsername` already identifies the account

In [40]:
cols_to_drop_insta = [
    'inputUrl', 'dimensionsHeight', 'dimensionsWidth',
    'musicInfo.should_mute_audio_reason', 'isCommentsDisabled',
    'displayUrl', 'originalHeight', 'originalWidth', 'alt', 'ownerId'
]
insta_df = insta_df.drop(columns=[c for c in cols_to_drop_insta if c in insta_df.columns])

print("Shape after column filtering:", insta_df.shape)
insta_df.head(3)


Shape after column filtering: (80, 32)


,id,type,shortCode,caption,hashtags,mentions,url,commentsCount,firstComment,latestComments,...,videoDuration,musicInfo.artist_name,musicInfo.song_name,musicInfo.uses_original_audio,musicInfo.should_mute_audio,musicInfo.audio_id,videoPlayCount,isPinned,paidPartnership,sponsors
0,3934596544011844684,Video,DaafrXcTjxM,Every win has a story. Every story begins long...,NaN,NaN,https://www.instagram.com/p/DaafrXcTjxM/,188,Good promotion 😂,"[{""id"": ""17878068531506204"", ""text"": ""Good pro...",...,42.92,muscleblaze,Original audio,True,False,2.731969e+16,181012.0,NaN,NaN,NaN
1,3930923218166072046,Sidecar,DaNcdcKH8Lu,"Before he carried the hopes of a nation, \nhe ...",NaN,parthaesthetix,https://www.instagram.com/p/DaNcdcKH8Lu/,43,❤️❤️,"[{""id"": ""18095779619196592"", ""text"": ""❤️❤️"", ""...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3929463792106173155,Video,DaIQn_uP3Lj,The records made him known.\n\nThe pursuit mad...,NaN,NaN,https://www.instagram.com/p/DaIQn_uP3Lj/,60,@rizwan.pathan_oo7,"[{""id"": ""18109613371790047"", ""text"": ""@rizwan....",...,37.32,muscleblaze,Original audio,True,False,2.829017e+16,184946.0,NaN,NaN,NaN


### 2.3 Data cleaning
- Remove duplicate posts (based on post `id`)
- Handle missing/invalid values in numeric columns
- Clean the caption text using our `clean_text()` function

In [65]:
before = len(insta_df)
insta_df = insta_df.drop_duplicates(subset='id')
print(f"Removed {before - len(insta_df)} duplicate post(s).")

insta_df['likesCount'] = pd.to_numeric(insta_df['likesCount'], errors='coerce').fillna(0)
insta_df['commentsCount'] = pd.to_numeric(insta_df['commentsCount'], errors='coerce').fillna(0)

insta_df['caption_clean'] = insta_df['caption'].apply(clean_text)
insta_df['firstComment_clean'] = insta_df['firstComment'].apply(clean_text)

insta_df[['caption','caption_clean', 'firstComment', 'firstComment_clean']].head(3)


Removed 0 duplicate post(s).


,caption,caption_clean,firstComment,firstComment_clean
0,Every win has a story. Every story begins long...,every win story every story begins long applau...,Good promotion 😂,good promotion
1,"Before he carried the hopes of a nation, \nhe ...",carried hopes nation carried weight becoming s...,❤️❤️,
2,The records made him known.\n\nThe pursuit mad...,records made known pursuit made story reveals ...,@rizwan.pathan_oo7,rizwanpathan_oo7


### 2.4 Row filtering
As an example filter, we keep only the posts that performed **above the median number of likes** — i.e. the better-performing half of the posts.

In [66]:
median_likes = insta_df['likesCount'].median()
insta_filtered = insta_df[insta_df['likesCount'] > median_likes]

print("Median likes:", median_likes)
print("Posts before filtering:", len(insta_df))
print("Posts after filtering (above median likes):", len(insta_filtered))
insta_filtered[['shortCode', 'likesCount', 'commentsCount']].head(5)


Median likes: 2350.0
Posts before filtering: 80
Posts after filtering (above median likes): 40


,shortCode,likesCount,commentsCount
0,DaafrXcTjxM,3106,188
1,DaNcdcKH8Lu,6605,43
2,DaIQn_uP3Lj,3964,60
4,DaxEl-5zgSy,9988,211
6,DZkF568z7Z4,3105,144


### 2.5 Download the cleaned dataset
We save the cleaned (but unfiltered) Instagram data as a CSV, so it can be used directly for EDA in the next experiment.

In [67]:
insta_df.to_csv('instagram_cleaned.csv', index=False)
print("Saved cleaned Instagram data to 'instagram_cleaned.csv'")

try:
    from google.colab import files
    files.download('instagram_cleaned.csv')
except ImportError:
    print("Not running in Colab — the file has been saved locally in this notebook's folder.")


Saved cleaned Instagram data to 'instagram_cleaned.csv'
Not running in Colab — the file has been saved locally in this notebook's folder.


### 2.6 Store cleaned data in SQLite

In [68]:
insta_df.to_sql('instagram_posts', sqlite_conn, if_exists='replace', index=False)
print(f"Stored {len(insta_df)} rows into the 'instagram_posts' table in SQLite.")


Stored 80 rows into the 'instagram_posts' table in SQLite.


### Store the same cleaned data in MongoDB

In [69]:
records = insta_df.where(pd.notnull(insta_df), None).to_dict('records')

collection = mongo_db['instagram_posts']
collection.delete_many({})
collection.insert_many(records)

print(f"Stored {len(records)} documents into the 'instagram_posts' collection in MongoDB.")


Stored 80 documents into the 'instagram_posts' collection in MongoDB.


---
## 3. YouTube Data — Cleaning, Filtering & Storage

### 3.1 Load and inspect the raw data

In [70]:
yt_df = pd.read_csv(YOUTUBE_CSV)
print("Shape:", yt_df.shape)
yt_df.head(3)


Shape: (100, 45)


,title,translatedTitle,type,id,url,thumbnailUrl,viewCount,date,likes,location,...,aiVideoSummary,order,commentsTurnedOff,fromYTUrl,isMonetized,hashtags,isMembersOnly,input,fromChannelListPage,isPaidContent
0,India’s Fastest Hurdler Ever | Tejas Shirse’s ...,NaN,video,saxrvRn3NOo,https://www.youtube.com/watch?v=saxrvRn3NOo,https://i.ytimg.com/vi/saxrvRn3NOo/maxresdefau...,1145522,2025-11-12T13:43:51.000Z,712,NaN,...,NaN,7,False,https://www.youtube.com/@MuscleBlaze/videos,NaN,[],False,https://www.youtube.com/@MuscleBlaze,videos,False
1,Biozyme Gold 100% Whey - Review by FRANK MEDRA...,NaN,video,EyXDAdnpyC0,https://www.youtube.com/watch?v=EyXDAdnpyC0,https://i.ytimg.com/vi/EyXDAdnpyC0/maxresdefau...,1378106,2025-02-27T10:10:05.000Z,65,NaN,...,NaN,20,False,https://www.youtube.com/@MuscleBlaze/videos,NaN,[],False,https://www.youtube.com/@MuscleBlaze,videos,False
2,Biozyme Whey - Reminder to Complete your Prote...,NaN,video,0dxaxkl3IyY,https://www.youtube.com/watch?v=0dxaxkl3IyY,https://i.ytimg.com/vi/0dxaxkl3IyY/maxresdefau...,312178,2025-03-13T11:30:00.000Z,45,NaN,...,NaN,19,False,https://www.youtube.com/@MuscleBlaze/videos,NaN,[],False,https://www.youtube.com/@MuscleBlaze,videos,False


### 3.2 Column filtering
The raw YouTube export turned out to have a lot of low-value columns once we actually inspected them:
- `thumbnailUrl`, `inputChannelUrl`, `fromChannelListPage` - scraper metadata with no analytical value
- All 14 `aboutChannelInfo__*` columns - these are **exact duplicates** of existing columns (e.g. `aboutChannelInfo__numberOfSubscribers` is identical to `numberOfSubscribers` for every row)
- `collaborators`, `aiVideoSummary`, `descriptionLinks`, `isMonetized`, `subtitles`, `aiVideoDescription` - 100% empty for every video
- `commentsTurnedOff`, `isMembersOnly`, `isPaidContent`, `isAgeRestricted`, `isChannelVerified` - constant (same value across all 100 videos), so they carry no information
- `translatedTitle`, `translatedText` - mostly empty, and where present, not actually translated (same language as the original)
- `order`, `fromYTUrl`, `input` - a row index and constant scraper URLs, no analytical value

In [71]:
cols_to_drop_yt = [
    'thumbnailUrl', 'inputChannelUrl', 'fromChannelListPage',
    'aboutChannelInfo__channelDescription', 'aboutChannelInfo__channelJoinedDate',
    'aboutChannelInfo__channelDescriptionLinks__text', 'aboutChannelInfo__channelDescriptionLinks__url',
    'aboutChannelInfo__channelLocation', 'aboutChannelInfo__channelUsername',
    'aboutChannelInfo__channelAvatarUrl', 'aboutChannelInfo__channelBannerUrl',
    'aboutChannelInfo__channelTotalVideos', 'aboutChannelInfo__channelTotalViews',
    'aboutChannelInfo__numberOfSubscribers', 'aboutChannelInfo__isChannelVerified',
    'aboutChannelInfo__channelName', 'aboutChannelInfo__channelUrl',
    'aboutChannelInfo__channelId', 'aboutChannelInfo__inputChannelUrl',
    'aboutChannelInfo__isAgeRestricted',
    'collaborators', 'aiVideoSummary', 'descriptionLinks', 'isMonetized', 'subtitles', 'aiVideoDescription',
    'commentsTurnedOff', 'isMembersOnly', 'isPaidContent', 'isAgeRestricted', 'isChannelVerified',
    'translatedTitle', 'translatedText',
    'order', 'fromYTUrl', 'input'
]
yt_df = yt_df.drop(columns=[c for c in cols_to_drop_yt if c in yt_df.columns])

print("Shape after column filtering:", yt_df.shape)
yt_df.head(3)


Shape after column filtering: (100, 26)


,title,type,id,url,viewCount,date,likes,location,channelName,channelUrl,...,channelAvatarUrl,channelBannerUrl,channelTotalVideos,channelTotalViews,numberOfSubscribers,aboutChannelInfo,duration,commentsCount,text,hashtags
0,India’s Fastest Hurdler Ever | Tejas Shirse’s ...,video,saxrvRn3NOo,https://www.youtube.com/watch?v=saxrvRn3NOo,1145522,2025-11-12T13:43:51.000Z,712,NaN,MuscleBlaze,https://www.youtube.com/channel/UCKlSCO8Tnr1_Q...,...,https://yt3.googleusercontent.com/3rWUhxMH_Agc...,https://yt3.googleusercontent.com/pwslaCDpj_u6...,1184,587247578,1060000,"{""channelDescription"": ""MuscleBlaze, with its ...",0:10:00,38,Every stride came with a ‘no.’\n\nNo money. No...,[]
1,Biozyme Gold 100% Whey - Review by FRANK MEDRA...,video,EyXDAdnpyC0,https://www.youtube.com/watch?v=EyXDAdnpyC0,1378106,2025-02-27T10:10:05.000Z,65,NaN,MuscleBlaze,https://www.youtube.com/channel/UCKlSCO8Tnr1_Q...,...,https://yt3.googleusercontent.com/3rWUhxMH_Agc...,https://yt3.googleusercontent.com/pwslaCDpj_u6...,1184,587247578,1060000,"{""channelDescription"": ""MuscleBlaze, with its ...",0:00:41,12,Stack your stock of 100% genuine supplements f...,[]
2,Biozyme Whey - Reminder to Complete your Prote...,video,0dxaxkl3IyY,https://www.youtube.com/watch?v=0dxaxkl3IyY,312178,2025-03-13T11:30:00.000Z,45,NaN,MuscleBlaze,https://www.youtube.com/channel/UCKlSCO8Tnr1_Q...,...,https://yt3.googleusercontent.com/3rWUhxMH_Agc...,https://yt3.googleusercontent.com/pwslaCDpj_u6...,1184,587247578,1060000,"{""channelDescription"": ""MuscleBlaze, with its ...",0:00:19,8,Stack your stock of 100% genuine supplements f...,[]


### 3.3 Data cleaning
- Drop fully blank rows (some scraper exports include empty spacer rows caused by nested fields like hashtags being flattened into extra columns)
- Remove duplicate videos (based on video `id`)
- Handle missing/invalid values in numeric columns
- Clean the video title text

In [72]:
# Drop fully blank rows (rows with no id/title - a common artifact in scraped CSV exports)
before = len(yt_df)
yt_df = yt_df.dropna(subset=['id', 'title'])
print(f"Removed {before - len(yt_df)} blank/empty row(s).")

before = len(yt_df)
yt_df = yt_df.drop_duplicates(subset='id')
print(f"Removed {before - len(yt_df)} duplicate video(s).")

yt_df['viewCount'] = pd.to_numeric(yt_df['viewCount'], errors='coerce').fillna(0)
yt_df['likes'] = pd.to_numeric(yt_df['likes'], errors='coerce').fillna(0)
yt_df['commentsCount'] = pd.to_numeric(yt_df['commentsCount'], errors='coerce').fillna(0)

yt_df['title_clean'] = yt_df['title'].apply(clean_text)

yt_df[['title', 'title_clean']].head(3)


Removed 0 blank/empty row(s).
Removed 0 duplicate video(s).


,title,title_clean
0,India’s Fastest Hurdler Ever | Tejas Shirse’s ...,indias fastest hurdler ever tejas shirses 1341...
1,Biozyme Gold 100% Whey - Review by FRANK MEDRA...,biozyme gold 100 whey review frank medrano pro...
2,Biozyme Whey - Reminder to Complete your Prote...,biozyme whey reminder complete protein goal


### 3.4 Row filtering
Here we filter for videos that performed **above the median view count**, to focus on the better-performing videos on the channel.

In [73]:
median_views = yt_df['viewCount'].median()
yt_filtered = yt_df[yt_df['viewCount'] > median_views]

print("Median views:", median_views)
print("Videos before filtering:", len(yt_df))
print("Videos after filtering (above median views):", len(yt_filtered))
yt_filtered[['title', 'viewCount', 'likes']].head(5)


Median views: 151339.5
Videos before filtering: 100
Videos after filtering (above median views): 50


,title,viewCount,likes
0,India’s Fastest Hurdler Ever | Tejas Shirse’s ...,1145522,712
1,Biozyme Gold 100% Whey - Review by FRANK MEDRA...,1378106,65
2,Biozyme Whey - Reminder to Complete your Prote...,312178,45
3,The Great Performance Sale 5.0 || 24 - 28 Marc...,177273,19
4,Biozyme Whey - Routine is Boring; but That's H...,312146,66


### 3.5 Download the cleaned dataset
We save the cleaned (but unfiltered) YouTube data as a CSV, so it can be used directly for EDA in the next experiment.

In [74]:
yt_df.to_csv('youtube_cleaned.csv', index=False)
print("Saved cleaned YouTube data to 'youtube_cleaned.csv'")

try:
    from google.colab import files
    files.download('youtube_cleaned.csv')
except ImportError:
    print("Not running in Colab — the file has been saved locally in this notebook's folder.")


Saved cleaned YouTube data to 'youtube_cleaned.csv'
Not running in Colab — the file has been saved locally in this notebook's folder.


### 3.6 Store cleaned data in SQLite

In [75]:
yt_df.to_sql('youtube_videos', sqlite_conn, if_exists='replace', index=False)
print(f"Stored {len(yt_df)} rows into the 'youtube_videos' table in SQLite.")


Stored 100 rows into the 'youtube_videos' table in SQLite.


### Store the same cleaned data in MongoDB

In [76]:
records = yt_df.where(pd.notnull(yt_df), None).to_dict('records')

collection = mongo_db['youtube_videos']
collection.delete_many({})
collection.insert_many(records)

print(f"Stored {len(records)} documents into the 'youtube_videos' collection in MongoDB.")


Stored 100 documents into the 'youtube_videos' collection in MongoDB.


---
## 4. Amazon Data — Cleaning, Filtering & Storage

### 4.1 Load and inspect the raw data

This is the full, untouched export from the Amazon scraper. Apify returns every nested detail of a product (images, variants, A+ content, review-level fields, etc.) as separate flattened columns, so this file has over a thousand columns — most of which aren't useful for our analysis.

In [77]:
amz_df = pd.read_csv(AMAZON_CSV)
print("Shape:", amz_df.shape)
amz_df.head(3)


Shape: (100, 1271)


,title,url,asin,originalAsin,price/value,price/currency,inStock,inStockText,listPrice/value,listPrice/currency,...,aPlusContent/modules/6/items/1/data/Price,aPlusContent/modules/6/items/1/data/Fish Oil,aPlusContent/modules/6/items/1/data/EPA,aPlusContent/modules/6/items/1/data/DHA,aPlusContent/modules/6/items/1/data/No Fishy Aftertaste,aPlusContent/modules/6/items/1/data/Mercury Free,aPlusContent/modules/3/items/3/title,aPlusContent/modules/3/items/3/text,aPlusContent/modules/3/items/3/image/name,aPlusContent/modules/3/items/3/image/url
0,"MuscleBlaze L-Glutamine Powder, Unflavoured (5...",https://www.amazon.in/dp/B01LW7X5MO,B01LW7X5MO,B01LW7X5MO,889.0,â‚¹,True,In stock,1029.0,â‚¹,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MuscleBlaze Koshaveda T-Surge Black (90 Tablet...,https://www.amazon.in/dp/B0BYJ5Y3S9,B0BYJ5Y3S9,B0BYJ5Y3S9,1159.0,â‚¹,True,In stock,1319.0,â‚¹,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MuscleBlaze Training Bag with Shoe Compartment...,https://www.amazon.in/dp/B0CYQ4FWQM,B0CYQ4FWQM,B0CYQ4FWQM,799.0,â‚¹,True,In stock,1299.0,â‚¹,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 4.2 Column filtering
As the first filtering step, we reduce the dataset down to the columns that are actually useful for our analysis — product identity, pricing, rating, and description. Most of the 1271 raw columns are empty or irrelevant for most rows (e.g. variant-specific or A+ content fields that only apply to a handful of products), so this is a much more aggressive form of column filtering than the small drops we did for Instagram and YouTube.

In [78]:
key_cols = ['title', 'url', 'asin', 'brand', 'price/value', 'stars', 'reviewsCount', 'description']
amz_df = amz_df[key_cols].copy()

print("Shape after selecting key columns:", amz_df.shape)
amz_df.head(3)


Shape after selecting key columns: (100, 8)


,title,url,asin,brand,price/value,stars,reviewsCount,description
0,"MuscleBlaze L-Glutamine Powder, Unflavoured (5...",https://www.amazon.in/dp/B01LW7X5MO,B01LW7X5MO,MuscleBlaze,889.0,4.1,1808.0,NaN
1,MuscleBlaze Koshaveda T-Surge Black (90 Tablet...,https://www.amazon.in/dp/B0BYJ5Y3S9,B0BYJ5Y3S9,MuscleBlaze,1159.0,4.0,451.0,NaN
2,MuscleBlaze Training Bag with Shoe Compartment...,https://www.amazon.in/dp/B0CYQ4FWQM,B0CYQ4FWQM,MuscleBlaze,799.0,4.1,849.0,NaN


### 4.3 Data cleaning
- Remove duplicate products (based on `asin`, Amazon's unique product ID)
- Convert price and rating columns to numeric, and drop rows with no title
- Clean the product description text

In [79]:
before = len(amz_df)
amz_df = amz_df.drop_duplicates(subset='asin')
print(f"Removed {before - len(amz_df)} duplicate product(s).")

amz_df['stars'] = pd.to_numeric(amz_df['stars'], errors='coerce')
amz_df['price/value'] = pd.to_numeric(amz_df['price/value'], errors='coerce')

before = len(amz_df)
amz_df = amz_df.dropna(subset=['title'])
print(f"Removed {before - len(amz_df)} row(s) with missing title.")

amz_df['description_clean'] = amz_df['description'].apply(clean_text)

amz_df[['title', 'description_clean']].head(3)


Removed 0 duplicate product(s).
Removed 0 row(s) with missing title.


,title,description_clean
0,"MuscleBlaze L-Glutamine Powder, Unflavoured (5...",
1,MuscleBlaze Koshaveda T-Surge Black (90 Tablet...,
2,MuscleBlaze Training Bag with Shoe Compartment...,


### 4.4 Row filtering
As an example, we filter for products with a **rating of 4.0 stars or higher**, to look specifically at well-rated products.

In [80]:
amz_filtered = amz_df[amz_df['stars'] >= 4.0]

print("Products before filtering:", len(amz_df))
print("Products after filtering (stars >= 4.0):", len(amz_filtered))
amz_filtered[['title', 'brand', 'stars', 'price/value']].head(5)


Products before filtering: 100
Products after filtering (stars >= 4.0): 77


,title,brand,stars,price/value
0,"MuscleBlaze L-Glutamine Powder, Unflavoured (5...",MuscleBlaze,4.1,889.0
1,MuscleBlaze Koshaveda T-Surge Black (90 Tablet...,MuscleBlaze,4.0,1159.0
2,MuscleBlaze Training Bag with Shoe Compartment...,MuscleBlaze,4.1,799.0
4,MuscleBlaze Half Sleeve Z-Verse T-Shirt,MuscleBlaze,4.2,999.0
5,MuscleBlaze Micronised Creatine Monohydrate Cr...,MuscleBlaze,4.2,549.0


### 4.5 Download the cleaned dataset
We save the cleaned (but unfiltered) Amazon data as a CSV, so it can be used directly for EDA in the next experiment.

In [81]:
amz_df.to_csv('amazon_cleaned.csv', index=False)
print("Saved cleaned Amazon data to 'amazon_cleaned.csv'")

try:
    from google.colab import files
    files.download('amazon_cleaned.csv')
except ImportError:
    print("Not running in Colab — the file has been saved locally in this notebook's folder.")


Saved cleaned Amazon data to 'amazon_cleaned.csv'
Not running in Colab — the file has been saved locally in this notebook's folder.


### 4.6 Store cleaned data in SQLite

In [82]:
amz_df.to_sql('amazon_products', sqlite_conn, if_exists='replace', index=False)
print(f"Stored {len(amz_df)} rows into the 'amazon_products' table in SQLite.")


Stored 100 rows into the 'amazon_products' table in SQLite.


### Store the same cleaned data in MongoDB

In [83]:
# Convert NaN values to None so they store cleanly as MongoDB nulls, and convert to a list of documents
records = amz_df.where(pd.notnull(amz_df), None).to_dict('records')

collection = mongo_db['amazon_products']
collection.delete_many({})  # clear old data so re-running the notebook doesn't create duplicates
collection.insert_many(records)

print(f"Stored {len(records)} documents into the 'amazon_products' collection in MongoDB.")


Stored 100 documents into the 'amazon_products' collection in MongoDB.


---
## 5. Verify data stored in SQLite

We confirm that all three cleaned datasets were stored correctly by querying SQLite directly.

In [84]:
with sqlite_conn as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]
    print("Tables in database:", tables)

    for table in ['instagram_posts', 'youtube_videos', 'amazon_products']:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        count = cursor.fetchone()[0]
        print(f"{table}: {count} rows")


Tables in database: ['instagram_posts', 'youtube_videos', 'amazon_products']
instagram_posts: 80 rows
youtube_videos: 100 rows
amazon_products: 100 rows


In [85]:
# Preview a few rows from each table directly from SQLite
print("Instagram sample:")
display(pd.read_sql("SELECT * FROM instagram_posts LIMIT 3", sqlite_conn))

print("YouTube sample:")
display(pd.read_sql("SELECT * FROM youtube_videos LIMIT 3", sqlite_conn))

print("Amazon sample:")
display(pd.read_sql("SELECT * FROM amazon_products LIMIT 3", sqlite_conn))


Instagram sample:


,id,type,shortCode,caption,hashtags,mentions,url,commentsCount,firstComment,latestComments,...,musicInfo.song_name,musicInfo.uses_original_audio,musicInfo.should_mute_audio,musicInfo.audio_id,videoPlayCount,isPinned,paidPartnership,sponsors,caption_clean,firstComment_clean
0,3934596544011844684,Video,DaafrXcTjxM,Every win has a story. Every story begins long...,None,None,https://www.instagram.com/p/DaafrXcTjxM/,188,Good promotion 😂,"[{""id"": ""17878068531506204"", ""text"": ""Good pro...",...,Original audio,1.0,0.0,2.731969e+16,181012.0,None,None,None,every win story every story begins long applau...,good promotion
1,3930923218166072046,Sidecar,DaNcdcKH8Lu,"Before he carried the hopes of a nation, \nhe ...",None,parthaesthetix,https://www.instagram.com/p/DaNcdcKH8Lu/,43,❤️❤️,"[{""id"": ""18095779619196592"", ""text"": ""❤️❤️"", ""...",...,None,NaN,NaN,NaN,NaN,None,None,None,carried hopes nation carried weight becoming s...,
2,3929463792106173155,Video,DaIQn_uP3Lj,The records made him known.\n\nThe pursuit mad...,None,None,https://www.instagram.com/p/DaIQn_uP3Lj/,60,@rizwan.pathan_oo7,"[{""id"": ""18109613371790047"", ""text"": ""@rizwan....",...,Original audio,1.0,0.0,2.829017e+16,184946.0,None,None,None,records made known pursuit made story reveals ...,rizwanpathan_oo7


YouTube sample:


,title,type,id,url,viewCount,date,likes,location,channelName,channelUrl,...,channelBannerUrl,channelTotalVideos,channelTotalViews,numberOfSubscribers,aboutChannelInfo,duration,commentsCount,text,hashtags,title_clean
0,India’s Fastest Hurdler Ever | Tejas Shirse’s ...,video,saxrvRn3NOo,https://www.youtube.com/watch?v=saxrvRn3NOo,1145522,2025-11-12T13:43:51.000Z,712,None,MuscleBlaze,https://www.youtube.com/channel/UCKlSCO8Tnr1_Q...,...,https://yt3.googleusercontent.com/pwslaCDpj_u6...,1184,587247578,1060000,"{""channelDescription"": ""MuscleBlaze, with its ...",0:10:00,38,Every stride came with a ‘no.’\n\nNo money. No...,[],indias fastest hurdler ever tejas shirses 1341...
1,Biozyme Gold 100% Whey - Review by FRANK MEDRA...,video,EyXDAdnpyC0,https://www.youtube.com/watch?v=EyXDAdnpyC0,1378106,2025-02-27T10:10:05.000Z,65,None,MuscleBlaze,https://www.youtube.com/channel/UCKlSCO8Tnr1_Q...,...,https://yt3.googleusercontent.com/pwslaCDpj_u6...,1184,587247578,1060000,"{""channelDescription"": ""MuscleBlaze, with its ...",0:00:41,12,Stack your stock of 100% genuine supplements f...,[],biozyme gold 100 whey review frank medrano pro...
2,Biozyme Whey - Reminder to Complete your Prote...,video,0dxaxkl3IyY,https://www.youtube.com/watch?v=0dxaxkl3IyY,312178,2025-03-13T11:30:00.000Z,45,None,MuscleBlaze,https://www.youtube.com/channel/UCKlSCO8Tnr1_Q...,...,https://yt3.googleusercontent.com/pwslaCDpj_u6...,1184,587247578,1060000,"{""channelDescription"": ""MuscleBlaze, with its ...",0:00:19,8,Stack your stock of 100% genuine supplements f...,[],biozyme whey reminder complete protein goal


Amazon sample:


,title,url,asin,brand,price/value,stars,reviewsCount,description,description_clean
0,"MuscleBlaze L-Glutamine Powder, Unflavoured (5...",https://www.amazon.in/dp/B01LW7X5MO,B01LW7X5MO,MuscleBlaze,889.0,4.1,1808.0,None,
1,MuscleBlaze Koshaveda T-Surge Black (90 Tablet...,https://www.amazon.in/dp/B0BYJ5Y3S9,B0BYJ5Y3S9,MuscleBlaze,1159.0,4.0,451.0,None,
2,MuscleBlaze Training Bag with Shoe Compartment...,https://www.amazon.in/dp/B0CYQ4FWQM,B0CYQ4FWQM,MuscleBlaze,799.0,4.1,849.0,None,


---
## 6. Verify data stored in MongoDB

Finally, we confirm the same cleaned datasets were also stored correctly in MongoDB by querying it directly.

In [86]:
print("Collections in database:", mongo_db.list_collection_names())

for coll_name in ['instagram_posts', 'youtube_videos', 'amazon_products']:
    count = mongo_db[coll_name].count_documents({})
    print(f"{coll_name}: {count} documents")


Collections in database: ['youtube_videos', 'amazon_products', 'instagram_posts']
instagram_posts: 80 documents
youtube_videos: 100 documents
amazon_products: 100 documents


In [87]:
# Preview one sample document from each collection
import pprint

for coll_name in ['instagram_posts', 'youtube_videos', 'amazon_products']:
    print(f"\nSample document from '{coll_name}':")
    pprint.pprint(mongo_db[coll_name].find_one())



Sample document from 'instagram_posts':
{'_id': ObjectId('6a7862e74abf493b5a029992'),
 'audioUrl': 'https://scontent-lga3-1.cdninstagram.com/o1/v/t2/f2/m78/AQOH9BGcTtMUuSBO7PT6XHXWxo5rZIr04uuh4pHrsUPYn1vfWNrZOF3Pwf6oJfCCxC6FlQ0VzLt0iZRYz7qwn36IS2JV67r6lfaEdCc.mp4?_nc_cat=111&_nc_sid=9ca052&_nc_ht=scontent-lga3-1.cdninstagram.com&_nc_ohc=kmIytoeLFlEQ7kNvwF6wy08&efg=eyJ2ZW5jb2RlX3RhZyI6ImlnLXhwdmRzLmNsaXBzLmlnd3d3LUMzLmRhc2hfbG5faGVhYWNfdmJyM19hdWRpbyIsInZpZGVvX2lkIjpudWxsLCJvaWxfdXJsZ2VuX2FwcF9pZCI6OTM2NjE5NzQzMzkyNDU5LCJjbGllbnRfbmFtZSI6ImlnIiwieHB2X2Fzc2V0X2lkIjoxNzQzNTY5MzQ2Nzk2MTc5LCJhc3NldF9hZ2VfZGF5cyI6MTQsInZpX3VzZWNhc2VfaWQiOjEwMDk5LCJkdXJhdGlvbl9zIjo0MiwiYml0cmF0ZSI6NTQyNDgsInVybGdlbl9zb3VyY2UiOiJ3d3cifQ%3D%3D&ccb=17-1&_nc_gid=Y1meqQG6opGaunqWdHtZVw&_nc_ss=7c689&_nc_zt=28&oh=00_AQCa3MFpbnjHAFAHktmeHXqkhIwBQImQZ_cQ7-QTsu38DA&oe=6A5FAC54',
 'caption': 'Every win has a story. Every story begins long before the '
            'applause.\n'
            '\n'
            'In Pursuit O

Your cleaned data for all three platforms is now stored in **both**:
- A `social_media.db` **SQLite** file (relational tables), and
- The `social_media_analytics` **MongoDB** database (non-relational collections)

You can open the `.db` file in a free tool like **DB Browser for SQLite** to see the relational tables, and **MongoDB Compass** to browse the MongoDB collections — useful for your "Snapshot of business data stored in relational database" and "Snapshot of business data stored in non relational database" screenshots respectively.